In [19]:
import pandas as pd
from collections import Counter
import numpy as np
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import re

In [2]:
df = pd.read_csv('/data/ephemeral/pro-nlp-finalproject-nlp-13/data/persona_data/AiHub_multiturn_3window.csv',usecols=['res','req'])

In [3]:
for col in df.columns:
    df[col] = df[col].str.replace('\n', ' ')

In [4]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')

In [ ]:
def quick_token_analysis(df, text_column='res'):
    texts = df[text_column].dropna().astype(str).tolist()
    
    # 전체 토큰화
    all_tokens = []
    for text in texts:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    token_counts = Counter(all_tokens)
    
    print(f"=== {text_column} 컬럼 토큰 분석 ===")
    print(f"총 샘플 수: {len(texts):,}")
    print(f"총 토큰 수: {len(all_tokens):,}")
    print(f"고유 토큰 수: {len(token_counts):,}")
    print(f"다양성 비율: {len(token_counts)/len(all_tokens)*100:.2f}%")
    
    # 집중도
    total = len(all_tokens)
    top10 = sum([cnt for _, cnt in token_counts.most_common(10)]) / total
    top100 = sum([cnt for _, cnt in token_counts.most_common(100)]) / total
    
    print(f"\n상위 10개 토큰 집중도: {top10*100:.2f}%")
    print(f"상위 100개 토큰 집중도: {top100*100:.2f}%")
    
    # 상위 토큰들
    print(f"\n가장 많이 사용된 토큰 20개:")
    for token_id, count in token_counts.most_common(20):
        token_str = tokenizer.decode([token_id])
        print(f"  '{token_str}': {count:,}회 ({count/total*100:.2f}%)")
    
    return token_counts

# 둘 다 분석
print("=" * 60)
print("응답(res) 분석")
print("=" * 60)
res_tokens = quick_token_analysis(df, 'res')

print("\n" + "=" * 60)
print("요청(req) 분석")
print("=" * 60)
req_tokens = quick_token_analysis(df, 'req')

응답(res) 분석
=== res 컬럼 토큰 분석 ===
총 샘플 수: 500,707
총 토큰 수: 10,180,639
고유 토큰 수: 7,242
다양성 비율: 0.07%

상위 10개 토큰 집중도: 13.44%
상위 100개 토큰 집중도: 47.25%

가장 많이 사용된 토큰 20개:
  '!': 212,473회 (2.09%)
  '.': 174,155회 (1.71%)
  '이': 158,335회 (1.56%)
  '?': 148,080회 (1.45%)
  '도': 138,257회 (1.36%)
  '을': 126,073회 (1.24%)
  '야': 111,902회 (1.10%)
  '아': 109,757회 (1.08%)
  '어': 96,570회 (0.95%)
  ',': 93,040회 (0.91%)
  ' �': 88,371회 (0.87%)
  ' ': 87,224회 (0.86%)
  '가': 81,173회 (0.80%)
  '에': 80,085회 (0.79%)
  ' 가': 74,600회 (0.73%)
  '지': 71,757회 (0.70%)
  ' 수': 71,625회 (0.70%)
  '는': 68,726회 (0.68%)
  '은': 68,719회 (0.67%)
  ' 거': 66,350회 (0.65%)

요청(req) 분석
=== req 컬럼 토큰 분석 ===
총 샘플 수: 500,707
총 토큰 수: 30,368,734
고유 토큰 수: 7,609
다양성 비율: 0.03%

상위 10개 토큰 집중도: 13.48%
상위 100개 토큰 집중도: 46.82%

가장 많이 사용된 토큰 20개:
  '!': 563,025회 (1.85%)
  '?': 543,293회 (1.79%)
  '.': 502,337회 (1.65%)
  '이': 488,330회 (1.61%)
  '도': 396,579회 (1.31%)
  '을': 347,675회 (1.14%)
  '야': 328,249회 (1.08%)
  ' ': 327,386회 (1.08%)
  ' �': 300,0

In [6]:
def check_ngram_patterns(df, text_column='res', n=2, top_k=15):
    ngrams = []
    
    for text in df[text_column].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i:i+n])
            ngrams.append(ngram)
    
    ngram_counts = Counter(ngrams)
    
    print(f"\n=== {n}-gram 패턴 (상위 {top_k}개) ===")
    print(f"총 {n}-gram: {len(ngrams):,}")
    print(f"고유 {n}-gram: {len(ngram_counts):,}")
    print(f"다양성: {len(ngram_counts)/len(ngrams)*100:.2f}%\n")
    
    for pattern, count in ngram_counts.most_common(top_k):
        decoded = tokenizer.decode(list(pattern))
        print(f"'{decoded}': {count:,}회")

In [65]:
check_ngram_patterns(df, 'res', n=2, top_k=10)
check_ngram_patterns(df, 'res', n=3, top_k=10)
check_ngram_patterns(df, 'res', n=5, top_k=10)
check_ngram_patterns(df, 'res', n=6, top_k=10)


=== 2-gram 패턴 (상위 10개) ===
총 2-gram: 9,679,932
고유 2-gram: 332,646
다양성: 3.44%

'아!': 35,964회
'맞아': 35,652회
' 거야': 33,135회
'ㅋㅋ': 29,163회
' 같아': 28,805회
'야?': 25,966회
' 것 같': 25,933회
'이야': 24,987회
' �': 23,243회
' 1': 20,199회

=== 3-gram 패턴 (상위 10개) ===
총 3-gram: 9,179,226
고유 3-gram: 1,714,574
다양성: 18.68%

' 것 같아': 25,143회
'맞아!': 24,545회
' 레': 15,268회
' 거야?': 14,731회
' 꼭': 12,686회
' ㅋㅋ': 10,849회
'그러면': 10,445회
' 곳이': 10,129회
' 볼 수': 9,787회
' 싶어': 9,581회

=== 5-gram 패턴 (상위 10개) ===
총 5-gram: 8,178,171
고유 5-gram: 4,460,998
다양성: 54.55%

' 좋을 것 같아': 5,430회
' 레알루': 3,672회
'��알루다가': 3,672회
' 렬루다가': 3,580회
' 꼭 가봐': 3,267회
' 레게노': 3,234회
' 수 있을 것 같아': 3,230회
'�� 가봐야': 2,928회
' 가보고 싶어': 2,922회
' 흥미로': 2,881회

=== 6-gram 패턴 (상위 10개) ===
총 6-gram: 7,678,217
고유 6-gram: 5,145,729
다양성: 67.02%

' 레알루다가': 3,672회
' 꼭 가봐야': 2,928회
' 레게노다': 2,833회
' 즐거운 시간 보내': 1,987회
' 흥미로워': 1,881회
' 맛있는 음식': 1,677회
'�거운 시간 보내자': 1,608회
'�� 가봐야겠다': 1,514회
' 가보고 싶어!': 1,506회
'면 좋을 것 같아': 1,424회


In [8]:
exact_dupes = df.duplicated(subset=['res'], keep=False)
n_exact_dupes = exact_dupes.sum()

print(f"전체 샘플: {len(df):,}개")
print(f"중복된 샘플: {n_exact_dupes:,}개 ({n_exact_dupes/len(df)*100:.2f}%)")
print(f"고유 샘플: {df['res'].nunique():,}개")

전체 샘플: 500,707개
중복된 샘플: 85,646개 (17.11%)
고유 샘플: 452,634개


In [9]:
df[exact_dupes].res

1080                      스쿠버다이빙은 3만원, 프리다이빙이랑 스노쿨링은 1만원이야.
2608      사우디아라비아 수도인 리야드와 기암괴석을 볼 수 있는 알 울라, 성지 메디나, 항구...
3452           국내 최고 모래조각 작가인 최지훈, 홍성렬 작가와 중국 장저우 작가가 제작했대.
3572                      스쿠버다이빙은 3만원, 프리다이빙이랑 스노쿨링은 1만원이야.
7329                                코레일관광개발 사이트에서 예약할 수 있어!
                                ...                        
498644                                      그렇게 말하니까 더 기대돼!
498652                                  짜짜? 그러면 나도 한 번 봐야겠다
498699                                              어떤 책이야?
498703                                            꼭 읽어봐야겠다!
500000                                     우리금융아트홀이 어디에 있어?
Name: res, Length: 85646, dtype: object

In [11]:
df[df.req=='오 굿굿! 언제 가자? 주말에 가면 좋을 것 같아 그래! 그럼 토요일에 가자']

,req,res
12993,오 굿굿! 언제 가자? 주말에 가면 좋을 것 같아 그래! 그럼 토요일에 가자,토요일은 날씨가 좋을 것 같아
22561,오 굿굿! 언제 가자? 주말에 가면 좋을 것 같아 그래! 그럼 토요일에 가자,토요일은 날씨가 좋을 것 같아
40046,오 굿굿! 언제 가자? 주말에 가면 좋을 것 같아 그래! 그럼 토요일에 가자,수성동계곡에는 석교와 목교가 있어
42282,오 굿굿! 언제 가자? 주말에 가면 좋을 것 같아 그래! 그럼 토요일에 가자,토요일에는 날씨가 좋을 것 같아
450237,오 굿굿! 언제 가자? 주말에 가면 좋을 것 같아 그래! 그럼 토요일에 가자,토요일에 만나자!


In [44]:
df[df.res=='어떤 책이야?']

,req,res,entropy
270190,헬롱! 헬롱! 뭐해? 책을 읽고 있어,어떤 책이야?,1.791759
352098,"훈민정음에 대해 들어봄? 응, 한글 만든 거지? 맞아! 그리고 최근에 훈민정음과 관...",어떤 책이야?,1.791759
386151,"오늘 뉴스 들었음? 응, 뭐 있었어? 한양대 교수가 책을 출간했대",어떤 책이야?,1.791759
389701,영화 어떤 거? 액션 영화야! 너는? 나는 책 읽고 있어 ㅋㅋ,어떤 책이야?,1.791759
390207,세계 뉴스 이해 어려워 맞아 ㅠㅠ 어렵게 느껴져 그런데 이번에 나온 책 알아?,어떤 책이야?,1.791759
394742,미술은 짜짜 어려워 나도 그렇게 생각해 그런데 최근에 미술 수업 책을 발견했어,어떤 책이야?,1.791759
400461,더위가 물러가서 다행이야 맞아 8월 장맛비로 시원해졌어 그래서 책을 꺼냈어,어떤 책이야?,1.791759
420820,그 작가는 만화가이기도 해 와 짜짜? 멋있네! 얼마 전에 책을 냈어,어떤 책이야?,1.791759
498699,한번 바꿔보면 기분이 좋아져! 레알? 나도 해봐야겠다 ㅎㅎ 그리고 이 책을 읽으면 ...,어떤 책이야?,1.791759


In [13]:
len(tokenizer)

151669

In [21]:
import re

def count_korean_number_special(tokenizer):
    """한글, 숫자, 특수문자만 세기"""
    
    vocab = tokenizer.get_vocab()
    total = len(vocab)
    
    # 카테고리별 카운트
    counts = {
        '한글_완성형': 0,      # 가-힣 포함
        '한글_자모': 0,        # ㄱ-ㅎ, ㅏ-ㅣ만
        '숫자': 0,
        '특수문자': 0,
        '공백/개행': 0,
        '바이트토큰': 0,       # <0xXX>
        '특수토큰': 0,         # <|endoftext|> 등
        '기타': 0
    }
    
    # 샘플 저장
    samples = {k: [] for k in counts.keys()}
    
    for token_str, token_id in vocab.items():
        try:
            decoded = tokenizer.decode([token_id])
        except:
            decoded = token_str
        
        categorized = False
        
        # 1. 특수 토큰
        if token_str.startswith('<') and token_str.endswith('>'):
            if '0x' in token_str:
                counts['바이트토큰'] += 1
                if len(samples['바이트토큰']) < 10:
                    samples['바이트토큰'].append((token_id, token_str, decoded))
            else:
                counts['특수토큰'] += 1
                if len(samples['특수토큰']) < 10:
                    samples['특수토큰'].append((token_id, token_str, decoded))
            categorized = True
        
        # 2. 공백/개행
        elif decoded.strip() == '' or decoded in ['\n', '\t', ' ']:
            counts['공백/개행'] += 1
            if len(samples['공백/개행']) < 10:
                samples['공백/개행'].append((token_id, token_str, repr(decoded)))
            categorized = True
        
        if not categorized:
            has_complete = bool(re.search(r'[가-힣]', decoded))
            has_jamo = bool(re.search(r'[ㄱ-ㅎㅏ-ㅣ]', decoded))
            
            # 3. 한글
            if has_complete:
                counts['한글_완성형'] += 1
                if len(samples['한글_완성형']) < 10:
                    samples['한글_완성형'].append((token_id, token_str, decoded))
                categorized = True
            
            elif has_jamo:
                counts['한글_자모'] += 1
                if len(samples['한글_자모']) < 10:
                    samples['한글_자모'].append((token_id, token_str, decoded))
                categorized = True
        
        # 4. 숫자만 (영어/한글 없이)
        if not categorized:
            if re.match(r'^[0-9\s]+$', decoded.strip()) and re.search(r'[0-9]', decoded):
                counts['숫자'] += 1
                if len(samples['숫자']) < 10:
                    samples['숫자'].append((token_id, token_str, decoded))
                categorized = True
        
        # 5. 특수문자만 (한글, 영어, 숫자, 공백 제외)
        if not categorized:
            if re.match(r'^[^\w가-힣ㄱ-ㅎㅏ-ㅣ\s]+$', decoded.strip()):
                counts['특수문자'] += 1
                if len(samples['특수문자']) < 10:
                    samples['특수문자'].append((token_id, token_str, decoded))
                categorized = True
        
        # 6. 기타
        if not categorized:
            counts['기타'] += 1
            if len(samples['기타']) < 10:
                samples['기타'].append((token_id, token_str, decoded))
    
    # 결과 출력
    print("=" * 80)
    print(f"Qwen Tokenizer Vocab 분석 (한글/숫자/특수문자)")
    print("=" * 80)
    print(f"\n전체 vocab 크기: {total:,}개\n")
    
    # 한글 합계
    korean_total = counts['한글_완성형'] + counts['한글_자모']
    
    print(f"{'한글 (전체)':15s}: {korean_total:7,}개 ({korean_total/total*100:5.2f}%)")
    print(f"  {'- 완성형':13s}: {counts['한글_완성형']:7,}개 ({counts['한글_완성형']/total*100:5.2f}%)")
    print(f"  {'- 자모':13s}: {counts['한글_자모']:7,}개 ({counts['한글_자모']/total*100:5.2f}%)")
    print(f"{'숫자':15s}: {counts['숫자']:7,}개 ({counts['숫자']/total*100:5.2f}%)")
    print(f"{'특수문자':15s}: {counts['특수문자']:7,}개 ({counts['특수문자']/total*100:5.2f}%)")
    print(f"{'공백/개행':15s}: {counts['공백/개행']:7,}개 ({counts['공백/개행']/total*100:5.2f}%)")
    print(f"{'바이트토큰':15s}: {counts['바이트토큰']:7,}개 ({counts['바이트토큰']/total*100:5.2f}%)")
    print(f"{'특수토큰':15s}: {counts['특수토큰']:7,}개 ({counts['특수토큰']/total*100:5.2f}%)")
    print(f"{'기타':15s}: {counts['기타']:7,}개 ({counts['기타']/total*100:5.2f}%)")
    
    # 샘플 출력
    print("\n" + "=" * 80)
    print("각 카테고리별 샘플")
    print("=" * 80)
    
    for category in ['한글_완성형', '한글_자모', '숫자', '특수문자']:
        sample_list = samples[category]
        if sample_list:
            print(f"\n[{category}] {counts[category]:,}개")
            for token_id, token_str, decoded in sample_list:
                print(f"  ID {token_id:6d}: '{decoded}'")
    
    return counts, samples

# 실행
counts, samples = count_korean_number_special(tokenizer)

Qwen Tokenizer Vocab 분석 (한글/숫자/특수문자)

전체 vocab 크기: 151,669개

한글 (전체)        :   3,532개 ( 2.33%)
  - 완성형        :   3,495개 ( 2.30%)
  - 자모         :      37개 ( 0.02%)
숫자             :      10개 ( 0.01%)
특수문자           :   9,322개 ( 6.15%)
공백/개행          :     441개 ( 0.29%)
바이트토큰          :       0개 ( 0.00%)
특수토큰           :      33개 ( 0.02%)
기타             : 138,331개 (91.21%)

각 카테고리별 샘플

[한글_완성형] 3,495개
  ID 136414: '행위'
  ID 133830: '생활'
  ID 151046: '챃'
  ID 144286: '큐'
  ID 126551: '는데'
  ID 126445: '프'
  ID 145229: '렙'
  ID 144042: '짐'
  ID 145455: '깍'
  ID 151024: '좐'

[한글_자모] 37개
  ID 149068: 'ㅔ'
  ID 147783: 'ㅃ'
  ID 146076: 'ㄲ'
  ID 144065: 'ㅠ'
  ID 151604: 'ㅖ'
  ID 145039: 'ㄷ'
  ID 146636: 'ㅗ'
  ID 145898: 'ㅉ'
  ID 145951: 'ㅌ'
  ID 145017: 'ㄱ'

[숫자] 10개
  ID     19: '4'
  ID     16: '1'
  ID     17: '2'
  ID     24: '9'
  ID     15: '0'
  ID     23: '8'
  ID     21: '6'
  ID     22: '7'
  ID     18: '3'
  ID     20: '5'

[특수문자] 9,322개
  ID    532: '}
'
  ID  54538: ' ..
'
  ID 1

In [24]:
def full_breakdown_with_english(df, text_col='res'):
    """영어 포함 전체 분류"""
    
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    used_tokens = set(all_tokens)
    
    korean = 0
    english = 0
    digit = 0
    special = 0
    space = 0
    etc = 0
    
    korean_samples = []
    english_samples = []
    
    for token_id in used_tokens:
        decoded = tokenizer.decode([token_id])
        
        has_korean = bool(re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        has_digit = bool(re.search(r'[0-9]', decoded))
        has_special = bool(re.search(r'[^\w가-힣ㄱ-ㅎㅏ-ㅣ\s]', decoded))
        
        categorized = False
        
        # 공백
        if decoded.strip() == '':
            space += 1
            categorized = True
        
        # 한글
        elif has_korean:
            korean += 1
            if len(korean_samples) < 20:
                korean_samples.append((token_id, decoded))
            categorized = True
        
        # 영어
        elif has_english and not has_korean:
            english += 1
            if len(english_samples) < 20:
                english_samples.append((token_id, decoded))
            categorized = True
        
        # 숫자
        elif has_digit and not has_korean and not has_english:
            digit += 1
            categorized = True
        
        # 특수문자
        elif has_special and not has_korean and not has_english:
            special += 1
            categorized = True
        
        if not categorized:
            etc += 1
    
    total = len(used_tokens)
    
    print(f"=== 사용된 토큰 전체 분류 ===")
    print(f"총 사용 토큰: {total:,}개\n")
    print(f"{'한글':10s}: {korean:5,}개 ({korean/total*100:5.2f}%)")
    print(f"{'영어':10s}: {english:5,}개 ({english/total*100:5.2f}%)")
    print(f"{'숫자':10s}: {digit:5,}개 ({digit/total*100:5.2f}%)")
    print(f"{'특수문자':10s}: {special:5,}개 ({special/total*100:5.2f}%)")
    print(f"{'공백':10s}: {space:5,}개 ({space/total*100:5.2f}%)")
    print(f"{'기타':10s}: {etc:5,}개 ({etc/total*100:5.2f}%)")
    
    print(f"\n검증: {korean+english+digit+special+space+etc} = {total}")
    
    # 샘플 확인
    print(f"\n=== 한글 토큰 샘플 20개 ===")
    for token_id, decoded in korean_samples:
        print(f"  ID {token_id:6d}: '{decoded}'")
    
    print(f"\n=== 영어 토큰 샘플 20개 ===")
    for token_id, decoded in english_samples:
        print(f"  ID {token_id:6d}: '{decoded}'")
    
    return {
        'korean': korean,
        'english': english,
        'digit': digit,
        'special': special,
        'space': space,
        'etc': etc
    }

# 실행
breakdown = full_breakdown_with_english(df, 'res')

=== 사용된 토큰 전체 분류 ===
총 사용 토큰: 7,242개

한글        : 2,610개 (36.04%)
영어        : 4,013개 (55.41%)
숫자        :    10개 ( 0.14%)
특수문자      :   453개 ( 6.26%)
공백        :     3개 ( 0.04%)
기타        :   153개 ( 2.11%)

검증: 7242 = 7242

=== 한글 토큰 샘플 20개 ===
  ID 131097: '새'
  ID 131110: '회사'
  ID 131120: '총'
  ID  98358: ' 종'
  ID 131131: '티'
  ID  32831: '성'
  ID 131137: '처럼'
  ID 131180: ' 여기'
  ID 131196: '브'
  ID 131193: '사항'
  ID 131219: ' 정도'
  ID 131226: '북'
  ID 131277: ' 내가'
  ID 131293: '림'
  ID 131303: '줄'
  ID 131342: '처리'
  ID 131344: '친'
  ID 131347: '충'
  ID 131355: '어요'
  ID 131366: ' 이러'

=== 영어 토큰 샘플 20개 ===
  ID     32: 'A'
  ID     33: 'B'
  ID     34: 'C'
  ID     35: 'D'
  ID     38: 'G'
  ID     36: 'E'
  ID     41: 'J'
  ID     42: 'K'
  ID     43: 'L'
  ID     44: 'M'
  ID     45: 'N'
  ID     46: 'O'
  ID     47: 'P'
  ID     49: 'R'
  ID     50: 'S'
  ID     51: 'T'
  ID     52: 'U'
  ID     53: 'V'
  ID     54: 'W'
  ID     56: 'Y'


In [25]:
# 영어 토큰 상위 빈도 확인
def check_english_tokens(df, text_col='res'):
    """영어 토큰이 뭔지 자세히 보기"""
    
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    token_counts = Counter(all_tokens)
    
    # 영어 토큰만 필터링
    english_tokens = []
    for token_id, count in token_counts.items():
        decoded = tokenizer.decode([token_id])
        
        has_korean = bool(re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        
        if has_english and not has_korean:
            english_tokens.append((token_id, decoded, count))
    
    # 빈도순 정렬
    english_tokens.sort(key=lambda x: x[2], reverse=True)
    
    print(f"=== 영어 토큰 상위 30개 ===")
    print(f"총 영어 토큰: {len(english_tokens):,}개\n")
    
    for i, (token_id, decoded, count) in enumerate(english_tokens[:30], 1):
        print(f"{i:2d}. ID {token_id:6d}: '{decoded:20s}' - {count:,}회")
    
    # 영어 토큰 총 사용 횟수
    total_english_count = sum(x[2] for x in english_tokens)
    total_tokens = len(all_tokens)
    
    print(f"\n영어 토큰 사용 빈도: {total_english_count:,}회 ({total_english_count/total_tokens*100:.2f}%)")

check_english_tokens(df, 'res')

=== 영어 토큰 상위 30개 ===
총 영어 토큰: 4,013개

 1. ID    730: ' K                  ' - 1,049회
 2. ID     38: 'G                   ' - 1,033회
 3. ID    479: ' G                  ' - 952회
 4. ID  14978: 'BJ                  ' - 876회
 5. ID     57: 'Z                   ' - 609회
 6. ID    386: ' M                  ' - 503회
 7. ID  16017: 'km                  ' - 470회
 8. ID     53: 'V                   ' - 465회
 9. ID     42: 'K                   ' - 448회
10. ID     76: 'm                   ' - 411회
11. ID   6159: ' CG                 ' - 336회
12. ID    328: ' S                  ' - 257회
13. ID    362: ' A                  ' - 253회
14. ID     35: 'D                   ' - 248회
15. ID  20192: ' DM                 ' - 221회
16. ID    451: ' N                  ' - 216회
17. ID   5487: 'ICE                 ' - 177회
18. ID   3994: 'FT                  ' - 175회
19. ID  96988: 'MZ                  ' - 173회
20. ID    425: ' B                  ' - 164회
21. ID  15653: 'TV                  ' - 164회
22. ID  21387

In [ ]:
# 이 토큰들이 어떤 맥락에서 쓰이는지 보기
def check_token_context(df, text_col='res', token_id=730):
    """특정 토큰이 쓰인 문장들 확인"""
    
    samples = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if token_id in tokens:
            samples.append(text)
            if len(samples) >= 10:
                break
    
    decoded_token = tokenizer.decode([token_id])
    print(f"=== Token ID {token_id} ('{decoded_token}')이 사용된 문장들 ===\n")
    
    for i, text in enumerate(samples, 1):
        print(f"{i:2d}. {text}")

# 상위 토큰들 확인
print("=" * 80)
check_token_context(df, 'res', token_id=730)   # ' K'
print("\n" + "=" * 80)
check_token_context(df, 'res', token_id=38)    # 'G'
print("\n" + "=" * 80)
check_token_context(df, 'res', token_id=14978) # 'BJ'
print("\n" + "=" * 80)
check_token_context(df, 'res', token_id=57) # 'BJ'
print("\n" + "=" * 80)
check_token_context(df, 'res', token_id=53) # 'BJ'
print("\n" + "=" * 80)
check_token_context(df, 'res', token_id=76) # 'BJ'

=== Token ID 730 (' K')이 사용된 문장들 ===

 1. 그러게. 심지어 이거 서울 뿐만 아니라 대구나 인천 이런 지역에도 꽤 있나봐. 요즘 K한옥 다시 재조명받는듯.
 2. 맞아. 서귀포 K POP 문화축제 같은 지역 축제랑 같이 하기도 하고 한다네. 그쪽은 진짜 초대박 예정이지. ㅎㅎ
 3. 당근이지 ㅋㅋ. 이번에 K-트래블 발전을 지원할 계획이라고 했대. 앞으로 다양한 지역 관광 혜택이 나올 것 같아. 야놀자 50% 숙박할인 받으면 진짜 가성비 최고일듯!
 4. 응. 그래서 서귀포 K-POP 문화축제 같은 지역 K-컬쳐 행사랑 연계해서 관광전세편도 유치할 계획이래.
 5. 무대행사, 부대행사, 프린지 페스티벌, K-POP 콘서트도 열린대!
 6. 헐, 진짜? K-POP 콘서트는 언제 열리는데? 그날만 갈까?
 7. 현재존은 K팝 콘서트나 퍼레이드 같은 공연 행사가 많아.
 8. 널 위한 K공포 축제가 있어. 한국 민속촌에서 하는 심야공포촌이라고 못 들어봤어?
 9. 와 처음에 분장을 너무 리얼하게 하신 분들이 많아서 놀랬음. Kijul 무섭게 느껴졌던 부분도 있었고 ㅠㅠ. 여기까지만 말함 ㅋㅋ. 더 말하면 완전 스포야! 직접 가서 보라고!ㅎㅎ
10. 와! 완전 K-공포 문화잖아? 너무 재밌겠는데? 나 벌써부터 막 떨린다. 입장료는 얼마야?

=== Token ID 38 ('G')이 사용된 문장들 ===

 1. G마켓이나 옥션에서 항공권을 구매하면 더 할인 받을 수 있어?
 2. 피렌체의 피렌체 대성당과 피사의 사원도 GBJG
 3. 사우디아라비아로 여행 GBJG!
 4. GOLD 회원권은 2,700만원, VIP 회원권은 5,000만원이야
 5. 맞아! 그리고 독립기념관도 있는데, 한 번 GBJG!
 6. 그래! 일단 대둔산과 월영산 출렁다리부터 GBJG
 7. 응! 꽃들이 만개한 국립세종수목원도 GBJG!
 8. 맞아! 청양 알품스도 GBJG
 9. 한 번 GBJG!
10. 그리고 묵호항 주변에 있는 도째비골스카이밸리도 GBJG!

=

In [34]:
check_token_context(df, 'res', token_id=76)

=== Token ID 76 ('m')이 사용된 문장들 ===

 1. 미래도시, 과학도시 하면 또 대전이잖아ㅋㅋ. 나노반도체, 우주항공, 바이오헬스, 국방산업 등 4대 핵심산업을 몰입형 실감 미디어 아트로 표현한 쇼가 있었대. 그리고 18m짜리 꿈돌이도 만들었음ㅋㅋ.
 2. 운전경력 2년 이상에 2종 보통면허증 있음 암오케지. 너희 부부는 운전 다 가능하잖냐. 대여 때 운전하기로 계약된 사람에 한해 보험 적용되고 여러 명이 운전할 경우 사전에 고지해야 함. 본인 차량을 화물차, 높이 3m로 설정하고 네비 이용해야 차량이 통과할 수 있는 구간으로 안내받으니 참고해.
 3. 루프탑 인피니티 풀이 있는 리조트야. 길이 20m, 폭 4.5m의 온수풀이 있어
 4. 지상 40m에서 미끄럼틀도 탈 수 있고, 디지털 아트세계도 볼 수 있어
 5. 시드니 하버브리지 클라이밍이라는 체험도 있던데, 134m 꼭대기까지 오르는 거래
 6. 산악지역 구릉형 코스로 해발 700m에 위치해 있어
 7. 시코쓰의 칼데라 호수는 깊이가 363m나 되는 거래
 8. 마나이 폭포는 높이가 17m로 박력이 있어!
 9. 그래! 그리고 제주 드림타워 복합리조트는 최고층이 62m야!
10. 응! 8초에 한 번씩 2.4m 파도타기도 가능해!


In [27]:
def check_space_prefix_tokens(df, text_col='res'):
    """공백으로 시작하는 토큰 분석"""
    
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    token_counts = Counter(all_tokens)
    used_tokens = set(all_tokens)
    
    # 공백으로 시작하는 토큰들
    space_prefix_korean = []
    space_prefix_english = []
    space_prefix_digit = []
    space_prefix_special = []
    
    for token_id in used_tokens:
        decoded = tokenizer.decode([token_id])
        
        # 공백으로 시작하는지 확인
        if decoded.startswith(' '):
            rest = decoded[1:]  # 공백 제거
            
            if re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', rest):
                space_prefix_korean.append((token_id, decoded, token_counts[token_id]))
            elif re.search(r'[a-zA-Z]', rest):
                space_prefix_english.append((token_id, decoded, token_counts[token_id]))
            elif re.search(r'[0-9]', rest):
                space_prefix_digit.append((token_id, decoded, token_counts[token_id]))
            else:
                space_prefix_special.append((token_id, decoded, token_counts[token_id]))
    
    # 정렬 (빈도순)
    space_prefix_korean.sort(key=lambda x: x[2], reverse=True)
    space_prefix_english.sort(key=lambda x: x[2], reverse=True)
    
    print("=" * 80)
    print("공백 접두어 토큰 분석")
    print("=" * 80)
    
    print(f"\n공백+한글: {len(space_prefix_korean):,}개")
    print("상위 30개:")
    for token_id, decoded, count in space_prefix_korean[:30]:
        print(f"  ID {token_id:6d}: '{decoded:20s}' - {count:,}회")
    
    print(f"\n공백+영어: {len(space_prefix_english):,}개")
    print("상위 30개:")
    for token_id, decoded, count in space_prefix_english[:30]:
        print(f"  ID {token_id:6d}: '{decoded:20s}' - {count:,}회")
    
    print(f"\n공백+숫자: {len(space_prefix_digit):,}개")
    if space_prefix_digit:
        for token_id, decoded, count in space_prefix_digit[:10]:
            print(f"  ID {token_id:6d}: '{decoded:20s}' - {count:,}회")
    
    # 재분류
    print("\n" + "=" * 80)
    print("재분류 결과")
    print("=" * 80)
    
    print(f"공백+한글: {len(space_prefix_korean):,}개")
    print(f"공백+영어: {len(space_prefix_english):,}개")
    print(f"공백+숫자: {len(space_prefix_digit):,}개")
    print(f"공백+기타: {len(space_prefix_special):,}개")
    
    return {
        'space_korean': space_prefix_korean,
        'space_english': space_prefix_english,
        'space_digit': space_prefix_digit
    }

# 실행
space_tokens = check_space_prefix_tokens(df, 'res')

공백 접두어 토큰 분석

공백+한글: 651개
상위 30개:
  ID  35509: ' 가                  ' - 74,600회
  ID  28733: ' 수                  ' - 71,625회
  ID 126352: ' 거                  ' - 66,350회
  ID 127353: ' 있어                 ' - 50,453회
  ID  60716: ' 해                  ' - 32,491회
  ID  78374: ' 같                  ' - 30,408회
  ID  54116: ' 기                  ' - 29,571회
  ID  71108: ' 것                  ' - 28,737회
  ID  54825: ' 그                  ' - 27,523회
  ID  35339: ' 있                  ' - 26,485회
  ID 142815: ' 여행                 ' - 26,425회
  ID  73518: ' 나                  ' - 24,421회
  ID 129273: ' 어떤                 ' - 22,940회
  ID 125953: ' 좋                  ' - 22,416회
  ID  56419: ' 전                  ' - 21,475회
  ID  64521: ' 있는                 ' - 20,834회
  ID 125466: ' 공                  ' - 20,703회
  ID  63332: ' 보                  ' - 20,128회
  ID 126366: ' 더                  ' - 18,754회
  ID  95617: ' 예                  ' - 18,337회
  ID 131050: ' 같이                 ' - 18,187회


In [28]:
def analyze_vocab_space_korean(tokenizer):
    """전체 vocab에서 공백+한국어 토큰 분석"""
    
    vocab = tokenizer.get_vocab()
    total_vocab = len(vocab)
    
    # 카테고리별 분류
    space_korean = []      # ' 가', ' 한국' 등
    space_english = []     # ' K', ' the' 등
    pure_korean = []       # '가', '한국' (공백 없이)
    pure_english = []      # 'K', 'the' (공백 없이)
    
    for token_str, token_id in vocab.items():
        try:
            decoded = tokenizer.decode([token_id])
        except:
            decoded = token_str
        
        has_korean = bool(re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        starts_with_space = decoded.startswith(' ')
        
        # 공백으로 시작하는 경우
        if starts_with_space:
            if has_korean:
                space_korean.append((token_id, token_str, decoded))
            elif has_english:
                space_english.append((token_id, token_str, decoded))
        # 공백 없이 시작
        else:
            if has_korean and not has_english:
                pure_korean.append((token_id, token_str, decoded))
            elif has_english and not has_korean:
                pure_english.append((token_id, token_str, decoded))
    
    # 결과 출력
    print("=" * 80)
    print("전체 Vocab 분석: 공백 접두어 기준")
    print("=" * 80)
    print(f"\n전체 vocab: {total_vocab:,}개\n")
    
    print(f"{'공백+한글':20s}: {len(space_korean):6,}개 ({len(space_korean)/total_vocab*100:5.2f}%)")
    print(f"{'순수 한글':20s}: {len(pure_korean):6,}개 ({len(pure_korean)/total_vocab*100:5.2f}%)")
    print(f"{'한글 전체':20s}: {len(space_korean)+len(pure_korean):6,}개 ({(len(space_korean)+len(pure_korean))/total_vocab*100:5.2f}%)")
    print()
    print(f"{'공백+영어':20s}: {len(space_english):6,}개 ({len(space_english)/total_vocab*100:5.2f}%)")
    print(f"{'순수 영어':20s}: {len(pure_english):6,}개 ({len(pure_english)/total_vocab*100:5.2f}%)")
    print(f"{'영어 전체':20s}: {len(space_english)+len(pure_english):6,}개 ({(len(space_english)+len(pure_english))/total_vocab*100:5.2f}%)")
    
    # 샘플 출력
    print("\n" + "=" * 80)
    print("공백+한글 샘플 30개")
    print("=" * 80)
    for token_id, token_str, decoded in space_korean[:30]:
        print(f"  ID {token_id:6d}: '{decoded}'")
    
    print("\n" + "=" * 80)
    print("순수 한글 샘플 30개")
    print("=" * 80)
    for token_id, token_str, decoded in pure_korean[:30]:
        print(f"  ID {token_id:6d}: '{decoded}'")
    
    return {
        'space_korean': len(space_korean),
        'pure_korean': len(pure_korean),
        'space_english': len(space_english),
        'pure_english': len(pure_english),
        'total_korean': len(space_korean) + len(pure_korean),
        'total_english': len(space_english) + len(pure_english)
    }

# 실행
vocab_stats = analyze_vocab_space_korean(tokenizer)

전체 Vocab 분석: 공백 접두어 기준

전체 vocab: 151,669개

공백+한글               :    667개 ( 0.44%)
순수 한글               :  2,865개 ( 1.89%)
한글 전체               :  3,532개 ( 2.33%)

공백+영어               : 45,201개 (29.80%)
순수 영어               : 48,687개 (32.10%)
영어 전체               : 93,888개 (61.90%)

공백+한글 샘플 30개
  ID 125640: ' 또'
  ID 139495: ' 작업'
  ID  81058: ' 처리'
  ID 128605: ' 대한'
  ID 137119: ' 있다고'
  ID 134808: ' 아닌'
  ID 126423: ' 실'
  ID 142180: ' 가운데'
  ID  75232: ' 객체'
  ID 129242: ' 재'
  ID  71015: ' 로'
  ID  45563: ' 값'
  ID 138098: ' 참여'
  ID 124830: ' 우리'
  ID 137769: ' 먼저'
  ID 136105: ' 모르'
  ID 130679: ' 위한'
  ID 141693: ' 겁'
  ID 126523: ' 아니'
  ID 141258: ' 있지만'
  ID 139764: ' 사진'
  ID 124973: ' 국'
  ID 142179: ' 가운'
  ID 135839: ' 받아'
  ID 143333: ' 국회'
  ID  83596: ' 받'
  ID 124657: ' 우'
  ID 139484: ' 그것을'
  ID 143522: ' 컴퓨터'
  ID 134583: ' 새'

순수 한글 샘플 30개
  ID 136559: '주시'
  ID 146057: '쏴'
  ID 150870: '깸'
  ID 144033: '칼'
  ID 145955: '앳'
  ID 148341: '븅'
  ID 145075: '왈'
  ID 145

In [31]:
def count_all_korean_tokens(df, text_col='res'):
    """자음/모음 포함한 전체 한글 토큰"""
    
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    used_tokens = set(all_tokens)
    
    # 4가지로 분류
    space_complete = 0  # ' 가', ' 한글' (공백+완성형)
    pure_complete = 0   # '가', '한글' (완성형만)
    space_jamo = 0      # ' ㄱ', ' ㅋ' (공백+자모)
    pure_jamo = 0       # 'ㄱ', 'ㅋ' (자모만)
    
    space_complete_list = []
    pure_complete_list = []
    space_jamo_list = []
    pure_jamo_list = []
    
    for token_id in used_tokens:
        decoded = tokenizer.decode([token_id])
        
        has_complete = bool(re.search(r'[가-힣]', decoded))
        has_jamo = bool(re.search(r'[ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        starts_with_space = decoded.startswith(' ')
        
        # 영어 섞인 건 제외
        if has_english:
            continue
        
        # 완성형 한글
        if has_complete:
            if starts_with_space:
                space_complete += 1
                space_complete_list.append((token_id, decoded))
            else:
                pure_complete += 1
                pure_complete_list.append((token_id, decoded))
        
        # 자모만 (완성형 없이)
        elif has_jamo:
            if starts_with_space:
                space_jamo += 1
                space_jamo_list.append((token_id, decoded))
            else:
                pure_jamo += 1
                pure_jamo_list.append((token_id, decoded))
    
    # 합계
    total_complete = space_complete + pure_complete
    total_jamo = space_jamo + pure_jamo
    total_korean = total_complete + total_jamo
    
    print("=" * 80)
    print("전체 한글 토큰 (자음/모음 포함)")
    print("=" * 80)
    print(f"\n총 사용 토큰: {len(used_tokens):,}개\n")
    
    print(f"{'완성형 한글':20s}: {total_complete:5,}개 ({total_complete/len(used_tokens)*100:5.2f}%)")
    print(f"  {'- 공백+완성형':18s}: {space_complete:5,}개")
    print(f"  {'- 순수 완성형':18s}: {pure_complete:5,}개")
    print()
    print(f"{'자음/모음':20s}: {total_jamo:5,}개 ({total_jamo/len(used_tokens)*100:5.2f}%)")
    print(f"  {'- 공백+자모':18s}: {space_jamo:5,}개")
    print(f"  {'- 순수 자모':18s}: {pure_jamo:5,}개")
    print()
    print(f"{'한글 전체':20s}: {total_korean:5,}개 ({total_korean/len(used_tokens)*100:5.2f}%)")
    
    # 샘플 출력
    print("\n" + "=" * 80)
    print("자음/모음 토큰 샘플")
    print("=" * 80)
    
    if space_jamo_list:
        print("\n공백+자모 샘플:")
        for token_id, decoded in space_jamo_list[:20]:
            print(f"  ID {token_id:6d}: '{decoded}'")
    
    if pure_jamo_list:
        print("\n순수 자모 샘플:")
        for token_id, decoded in pure_jamo_list[:20]:
            print(f"  ID {token_id:6d}: '{decoded}'")
    
    return {
        'space_complete': space_complete,
        'pure_complete': pure_complete,
        'space_jamo': space_jamo,
        'pure_jamo': pure_jamo,
        'total_korean': total_korean
    }

# 실행
korean_full = count_all_korean_tokens(df, 'res')

전체 한글 토큰 (자음/모음 포함)

총 사용 토큰: 7,242개

완성형 한글              : 2,585개 (35.69%)
  - 공백+완성형          :   651개
  - 순수 완성형          : 1,934개

자음/모음               :    25개 ( 0.35%)
  - 공백+자모           :     0개
  - 순수 자모           :    25개

한글 전체               : 2,610개 (36.04%)

자음/모음 토큰 샘플

순수 자모 샘플:
  ID 144030: 'ㅎ'
  ID 144065: 'ㅠ'
  ID 144410: 'ㅋ'
  ID 144579: 'ㅜ'
  ID 144681: 'ㅇ'
  ID 144724: 'ㅡ'
  ID 145017: 'ㄱ'
  ID 145039: 'ㄷ'
  ID 145089: 'ㄴ'
  ID 145162: 'ㅣ'
  ID 145220: 'ㅅ'
  ID 145417: 'ㅁ'
  ID 145442: 'ㄹ'
  ID 145506: 'ㅂ'
  ID 145697: 'ㅈ'
  ID 145826: 'ㅍ'
  ID 145898: 'ㅉ'
  ID 145951: 'ㅌ'
  ID 146076: 'ㄲ'
  ID 146457: 'ㅊ'


In [36]:
def analyze_top_frequent_korean(df, text_col='res', top_n=100):
    """상위 빈도 토큰 중 한국어 비율 (공백+한국어 포함)"""
    
    # 모든 토큰 수집
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    token_counts = Counter(all_tokens)
    
    # 상위 N개
    top_tokens = token_counts.most_common(top_n)
    
    # 카테고리 분류
    korean_tokens = []
    english_tokens = []
    digit_tokens = []
    special_tokens = []
    etc_tokens = []
    
    for token_id, count in top_tokens:
        decoded = tokenizer.decode([token_id])
        
        # 한글 (완성형 + 자모)
        has_korean = bool(re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        has_digit = bool(re.search(r'[0-9]', decoded))
        
        # 한글 포함 (공백 상관없이, 영어 없으면 한글로 분류)
        if has_korean and not has_english:
            korean_tokens.append((token_id, decoded, count))
        elif has_english and not has_korean:
            english_tokens.append((token_id, decoded, count))
        elif has_digit:
            digit_tokens.append((token_id, decoded, count))
        elif re.match(r'^[^\w가-힣ㄱ-ㅎㅏ-ㅣ\s]+$', decoded.strip()):
            special_tokens.append((token_id, decoded, count))
        else:
            etc_tokens.append((token_id, decoded, count))
    
    # 빈도 합계
    korean_freq = sum(x[2] for x in korean_tokens)
    english_freq = sum(x[2] for x in english_tokens)
    digit_freq = sum(x[2] for x in digit_tokens)
    special_freq = sum(x[2] for x in special_tokens)
    etc_freq = sum(x[2] for x in etc_tokens)
    
    total_top_freq = sum(x[1] for x in top_tokens)
    total_all_tokens = len(all_tokens)
    
    print("=" * 80)
    print(f"상위 {top_n}개 빈도 토큰 분석 (공백+한국어 포함)")
    print("=" * 80)
    
    print(f"\n=== 토큰 종류 개수 ===")
    print(f"한글      : {len(korean_tokens):3d}개 ({len(korean_tokens)/top_n*100:5.1f}%)")
    print(f"영어      : {len(english_tokens):3d}개 ({len(english_tokens)/top_n*100:5.1f}%)")
    print(f"숫자      : {len(digit_tokens):3d}개 ({len(digit_tokens)/top_n*100:5.1f}%)")
    print(f"특수문자   : {len(special_tokens):3d}개 ({len(special_tokens)/top_n*100:5.1f}%)")
    print(f"기타      : {len(etc_tokens):3d}개 ({len(etc_tokens)/top_n*100:5.1f}%)")
    
    print(f"\n=== 실제 사용 빈도 (상위 {top_n}개 토큰의 총 출현 횟수) ===")
    print(f"한글      : {korean_freq:10,}회 ({korean_freq/total_top_freq*100:5.2f}%)")
    print(f"영어      : {english_freq:10,}회 ({english_freq/total_top_freq*100:5.2f}%)")
    print(f"숫자      : {digit_freq:10,}회 ({digit_freq/total_top_freq*100:5.2f}%)")
    print(f"특수문자   : {special_freq:10,}회 ({special_freq/total_top_freq*100:5.2f}%)")
    print(f"기타      : {etc_freq:10,}회 ({etc_freq/total_top_freq*100:5.2f}%)")
    
    print(f"\n상위 {top_n}개 토큰이 전체의 {total_top_freq/total_all_tokens*100:.2f}% 차지")
    
    # 한글 토큰 상세
    print("\n" + "=" * 80)
    print(f"상위 {top_n}개 중 한글 토큰 전체 목록 ({len(korean_tokens)}개)")
    print("=" * 80)
    for i, (token_id, decoded, count) in enumerate(korean_tokens, 1):
        # 공백 표시
        display = decoded.replace(' ', '␣')
        print(f"{i:3d}. ID {token_id:6d}: '{display:20s}' - {count:,}회")
    
    # 영어 토큰도 출력
    print("\n" + "=" * 80)
    print(f"상위 {top_n}개 중 영어 토큰 전체 목록 ({len(english_tokens)}개)")
    print("=" * 80)
    if len(english_tokens) > 0:
        for i, (token_id, decoded, count) in enumerate(english_tokens, 1):
            display = decoded.replace(' ', '␣')
            print(f"{i:3d}. ID {token_id:6d}: '{display:20s}' - {count:,}회")
    else:
        print("(영어 토큰 없음)")
    
    # 특수문자도 출력
    print("\n" + "=" * 80)
    print(f"상위 {top_n}개 중 특수문자 토큰 ({len(special_tokens)}개)")
    print("=" * 80)
    if len(special_tokens) > 0:
        for i, (token_id, decoded, count) in enumerate(special_tokens, 1):
            display = repr(decoded)  # 특수문자는 repr로
            print(f"{i:3d}. ID {token_id:6d}: {display:20s} - {count:,}회")
    else:
        print("(특수문자 토큰 없음)")
    
    return {
        'korean_count': len(korean_tokens),
        'english_count': len(english_tokens),
        'korean_freq': korean_freq,
        'english_freq': english_freq,
        'korean_tokens': korean_tokens,
        'english_tokens': english_tokens,
        'special_tokens': special_tokens
    }

# 실행
result = analyze_top_frequent_korean(df, 'res', top_n=100)

상위 100개 빈도 토큰 분석 (공백+한국어 포함)

=== 토큰 종류 개수 ===
한글      :  74개 ( 74.0%)
영어      :   0개 (  0.0%)
숫자      :   3개 (  3.0%)
특수문자   :  22개 ( 22.0%)
기타      :   1개 (  1.0%)

=== 실제 사용 빈도 (상위 100개 토큰의 총 출현 횟수) ===
한글      :  3,365,983회 (69.97%)
영어      :          0회 ( 0.00%)
숫자      :     84,998회 ( 1.77%)
특수문자   :  1,272,291회 (26.45%)
기타      :     87,224회 ( 1.81%)

상위 100개 토큰이 전체의 47.25% 차지

상위 100개 중 한글 토큰 전체 목록 (74개)
  1. ID  12802: '이                   ' - 158,335회
  2. ID  47985: '도                   ' - 138,257회
  3. ID  17877: '을                   ' - 126,073회
  4. ID  89659: '야                   ' - 111,902회
  5. ID  52959: '아                   ' - 109,757회
  6. ID  31079: '어                   ' - 96,570회
  7. ID  19969: '가                   ' - 81,173회
  8. ID  19391: '에                   ' - 80,085회
  9. ID  35509: '␣가                  ' - 74,600회
 10. ID  21329: '지                   ' - 71,757회
 11. ID  28733: '␣수                  ' - 71,625회
 12. ID  16560: '는                   ' -

In [37]:
result = analyze_top_frequent_korean(df, 'res', top_n=200)

상위 200개 빈도 토큰 분석 (공백+한국어 포함)

=== 토큰 종류 개수 ===
한글      : 146개 ( 73.0%)
영어      :   0개 (  0.0%)
숫자      :   4개 (  2.0%)
특수문자   :  49개 ( 24.5%)
기타      :   1개 (  0.5%)

=== 실제 사용 빈도 (상위 200개 토큰의 총 출현 횟수) ===
한글      :  4,519,969회 (70.56%)
영어      :          0회 ( 0.00%)
숫자      :     97,306회 ( 1.52%)
특수문자   :  1,701,310회 (26.56%)
기타      :     87,224회 ( 1.36%)

상위 200개 토큰이 전체의 62.92% 차지

상위 200개 중 한글 토큰 전체 목록 (146개)
  1. ID  12802: '이                   ' - 158,335회
  2. ID  47985: '도                   ' - 138,257회
  3. ID  17877: '을                   ' - 126,073회
  4. ID  89659: '야                   ' - 111,902회
  5. ID  52959: '아                   ' - 109,757회
  6. ID  31079: '어                   ' - 96,570회
  7. ID  19969: '가                   ' - 81,173회
  8. ID  19391: '에                   ' - 80,085회
  9. ID  35509: '␣가                  ' - 74,600회
 10. ID  21329: '지                   ' - 71,757회
 11. ID  28733: '␣수                  ' - 71,625회
 12. ID  16560: '는                   ' 

In [39]:
# 한글 토큰의 빈도 분포 확인
def analyze_korean_token_distribution(df, text_col='res'):
    """한글 토큰들이 얼마나 골고루 쓰였는지 확인"""
    
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    token_counts = Counter(all_tokens)
    
    # 한글 토큰만 추출
    korean_token_counts = []
    
    for token_id, count in token_counts.items():
        decoded = tokenizer.decode([token_id])
        has_korean = bool(re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        
        if has_korean and not has_english:
            korean_token_counts.append((token_id, decoded, count))
    
    # 빈도순 정렬
    korean_token_counts.sort(key=lambda x: x[2], reverse=True)
    
    total_korean_tokens = len(korean_token_counts)
    total_korean_freq = sum(x[2] for x in korean_token_counts)
    
    # 상위 N개가 차지하는 비율
    top10_freq = sum(x[2] for x in korean_token_counts[:10])
    top50_freq = sum(x[2] for x in korean_token_counts[:50])
    top100_freq = sum(x[2] for x in korean_token_counts[:100])
    top200_freq = sum(x[2] for x in korean_token_counts[:200])
    
    print("=" * 80)
    print("한글 토큰 빈도 집중도 분석")
    print("=" * 80)
    print(f"\n총 한글 토큰 종류: {total_korean_tokens:,}개")
    print(f"총 한글 토큰 빈도: {total_korean_freq:,}회\n")
    
    print(f"상위 10개 한글 토큰: {top10_freq:,}회 ({top10_freq/total_korean_freq*100:5.2f}%)")
    print(f"상위 50개 한글 토큰: {top50_freq:,}회 ({top50_freq/total_korean_freq*100:5.2f}%)")
    print(f"상위 100개 한글 토큰: {top100_freq:,}회 ({top100_freq/total_korean_freq*100:5.2f}%)")
    print(f"상위 200개 한글 토큰: {top200_freq:,}회 ({top200_freq/total_korean_freq*100:5.2f}%)")
    
    # 상위 한글 토큰 출력
    print("\n" + "=" * 80)
    print("가장 많이 사용된 한글 토큰 30개")
    print("=" * 80)
    for i, (token_id, decoded, count) in enumerate(korean_token_counts[:30], 1):
        display = decoded.replace(' ', '␣')
        print(f"{i:3d}. '{display:15s}' - {count:,}회 ({count/total_korean_freq*100:.2f}%)")
    
    # 하위 토큰들
    print("\n" + "=" * 80)
    print("거의 안 쓰인 한글 토큰 (1-5회만 사용)")
    print("=" * 80)
    
    rare_tokens = [x for x in korean_token_counts if x[2] <= 5]
    print(f"5회 이하 사용: {len(rare_tokens):,}개 ({len(rare_tokens)/total_korean_tokens*100:.1f}%)")
    
    if rare_tokens:
        print("\n예시 20개:")
        for token_id, decoded, count in rare_tokens[:20]:
            display = decoded.replace(' ', '␣')
            print(f"  '{display:15s}' - {count}회")
    
    return korean_token_counts

# 실행
korean_dist = analyze_korean_token_distribution(df, 'res')

한글 토큰 빈도 집중도 분석

총 한글 토큰 종류: 2,610개
총 한글 토큰 빈도: 7,702,061회

상위 10개 한글 토큰: 1,048,509회 (13.61%)
상위 50개 한글 토큰: 2,813,466회 (36.53%)
상위 100개 한글 토큰: 3,858,597회 (50.10%)
상위 200개 한글 토큰: 5,072,144회 (65.85%)

가장 많이 사용된 한글 토큰 30개
  1. '이              ' - 158,335회 (2.06%)
  2. '도              ' - 138,257회 (1.80%)
  3. '을              ' - 126,073회 (1.64%)
  4. '야              ' - 111,902회 (1.45%)
  5. '아              ' - 109,757회 (1.43%)
  6. '어              ' - 96,570회 (1.25%)
  7. '가              ' - 81,173회 (1.05%)
  8. '에              ' - 80,085회 (1.04%)
  9. '␣가             ' - 74,600회 (0.97%)
 10. '지              ' - 71,757회 (0.93%)
 11. '␣수             ' - 71,625회 (0.93%)
 12. '는              ' - 68,726회 (0.89%)
 13. '은              ' - 68,719회 (0.89%)
 14. '␣거             ' - 66,350회 (0.86%)
 15. 'ㅋ              ' - 64,701회 (0.84%)
 16. '의              ' - 59,836회 (0.78%)
 17. '서              ' - 58,874회 (0.76%)
 18. '고              ' - 58,605회 (0.76%)
 19. '나              ' - 58,598회 (0.76

In [42]:
def check_response_diversity(df, text_col='res'):
    """응답 다양성 확인"""
    
    total = len(df)
    unique = df[text_col].nunique()
    
    diversity_ratio = unique / total * 100
    avg_repeat = total / unique
    
    print("=" * 80)
    print("응답 다양성 분석")
    print("=" * 80)
    print(f"\n총 샘플: {total:,}개")
    print(f"고유 응답: {unique:,}개")
    print(f"다양성 비율: {diversity_ratio:.2f}%")
    print(f"평균 반복: {avg_repeat:.1f}회\n")
    
    # 가장 많이 반복된 응답
    print("가장 많이 반복된 응답 20개:")
    for i, (text, count) in enumerate(df[text_col].value_counts().head(20).items(), 1):
        print(f"{i:2d}. {count:4,}회: {text[:80]}")
    
    # 반복 횟수 분포
    repeat_counts = df[text_col].value_counts()
    
    once = (repeat_counts == 1).sum()
    twice_to_5 = ((repeat_counts >= 2) & (repeat_counts <= 5)).sum()
    six_to_10 = ((repeat_counts >= 6) & (repeat_counts <= 10)).sum()
    eleven_to_50 = ((repeat_counts >= 11) & (repeat_counts <= 50)).sum()
    over_50 = (repeat_counts > 50).sum()
    
    print(f"\n반복 횟수 분포:")
    print(f"  1회만: {once:,}개 ({once/unique*100:.1f}%)")
    print(f"  2-5회: {twice_to_5:,}개 ({twice_to_5/unique*100:.1f}%)")
    print(f"  6-10회: {six_to_10:,}개 ({six_to_10/unique*100:.1f}%)")
    print(f"  11-50회: {eleven_to_50:,}개 ({eleven_to_50/unique*100:.1f}%)")
    print(f"  50회 초과: {over_50:,}개 ({over_50/unique*100:.1f}%)")
    
    # 위험도 평가
    print("\n" + "=" * 80)
    print("과적합 위험도 평가")
    print("=" * 80)
    
    if diversity_ratio > 80:
        print("🟢 다양성 우수 (80%+)")
    elif diversity_ratio > 60:
        print("🟡 다양성 보통 (60-80%)")
    elif diversity_ratio > 40:
        print("🟠 다양성 낮음 (40-60%) - 중복 제거 권장")
    else:
        print("🔴 다양성 매우 낮음 (40% 미만) - 심각한 과적합 위험")
    
    if avg_repeat < 2:
        print("🟢 평균 반복 낮음")
    elif avg_repeat < 5:
        print("🟡 평균 반복 보통")
    elif avg_repeat < 10:
        print("🟠 평균 반복 높음")
    else:
        print("🔴 평균 반복 매우 높음 - 심각한 문제")
    
    return {
        'diversity_ratio': diversity_ratio,
        'avg_repeat': avg_repeat,
        'unique': unique
    }

# 실행
check_response_diversity(df, 'res')


응답 다양성 분석

총 샘플: 500,707개
고유 응답: 452,634개
다양성 비율: 90.40%
평균 반복: 1.1회

가장 많이 반복된 응답 20개:
 1.  128회: 너무 기대돼!
 2.  124회: 응! 기대돼!
 3.   99회: 잘 다녀와!
 4.   96회: ㅇㅋㅇㅋ 기대돼!
 5.   93회: 응, 기대돼!
 6.   92회: 그래! 기대돼!
 7.   85회: 기대돼!
 8.   82회: 그래! 같이 가자!
 9.   76회: 그래! 같이 가서 즐거운 시간 보내자!
10.   68회: 나도 기대해!
11.   61회: 나도 기대돼!
12.   61회: 그래! 같이 가자
13.   60회: 그럼 같이 가자!
14.   60회: 그 책은 어떤 내용이야?
15.   52회: 굿굿! 언제 가면 좋을까?
16.   47회: 그래! 그럼 토요일에 가자
17.   44회: 그럼 다음에 같이 가자!
18.   41회: 응, 꼭 가보는 걸 추천해!
19.   41회: 나도 같이 가자!
20.   41회: 안뇽!

반복 횟수 분포:
  1회만: 415,061개 (91.7%)
  2-5회: 36,891개 (8.2%)
  6-10회: 441개 (0.1%)
  11-50회: 226개 (0.0%)
  50회 초과: 15개 (0.0%)

과적합 위험도 평가
🟢 다양성 우수 (80%+)
🟢 평균 반복 낮음


{'diversity_ratio': 90.39897584815073,
 'avg_repeat': 1.1062072226125301,
 'unique': 452634}

In [43]:
from scipy.stats import entropy
from collections import Counter

def token_entropy_diverse_sampling(df, text_col='res', target_size=100000):
    """토큰 엔트로피 높은 샘플 우선 선택"""
    
    def calculate_entropy(text):
        """텍스트의 토큰 분포 엔트로피"""
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) == 0:
            return 0
        
        token_counts = Counter(tokens)
        probs = np.array(list(token_counts.values())) / len(tokens)
        return entropy(probs)
    
    # 각 샘플의 엔트로피 계산
    print("엔트로피 계산 중...")
    df['entropy'] = df[text_col].apply(calculate_entropy)
    
    # 엔트로피 높은 순 정렬 (다양한 토큰 사용한 샘플)
    df_sorted = df.sort_values('entropy', ascending=False)
    
    # 상위 선택
    sampled = df_sorted.head(target_size)
    
    print(f"평균 엔트로피:")
    print(f"  원본: {df['entropy'].mean():.3f}")
    print(f"  샘플링 후: {sampled['entropy'].mean():.3f}")
    
    return sampled.drop('entropy', axis=1)

# 사용
diverse_df = token_entropy_diverse_sampling(df, target_size=100000)

엔트로피 계산 중...
평균 엔트로피:
  원본: 2.835
  샘플링 후: 3.415


In [45]:
diverse_df

,req,res
8929,"글쿤, 열차를 교통수단으로써뿐 아니라 특별하게 이용할 수 있어서 조흐네. 우주센터가...",숙박 코스별로 다른데 먼저 마리안느 앤 마가렛 나눔연수원은 힐링 체험 위주로 진행된...
476096,시간은 딱 좋네! 근데 나 클래식공연은 처음인데 조는 거 아닌가 몰라 이번 공연 컨...,"응, 첫 곡은 타악기와 금관악기의 사운드가 돋보이는 코플랜드의 보통 사람을 위한 팡..."
242040,"아 정말? 나 파랑새 진짜 좋아했는데, 그럼 파랑새 얘기를 리메이크 한건가? 아니,...",20여 년이 지난 뒤의 틸틸은 나이는 들었지만 조금도 어른이 되지 못한 청년이야. ...
484555,"몰랐는데, 진정한 스위트 홈이네. 하긴 주변 보니 홈루덴스족이 심심찮게 보이던데, ...","이젠 IT, 식품, 가전, 콘텐츠 등 다양한 업계들이 소비 트렌드를 주목하고 풀충 ..."
7965,와~ 과거 존이 갬성 폭발할 것 같네. 거긴 어떤 행사를 하나? 대전역부터 한방특화...,과거존 끝에서 하얀 안개가 뿜어져 나오는 원형 터널을 지나면 'D-컬쳐 꿀잼대전 2...
...,...,...
480220,맞아! 그리고 선정된 작품은 최대 3년간 다년도 연속 지원도 받을 수 있대 짜짜 멋...,맞아! 그러면 우리도 언젠가 전통예술을 감상하러 가보는 건 어떰?
159713,"렬루? 그렇게 많이 구매하다니 머단해! 응! 등산은 건강에도 좋고, 자연을 느낄 수...",그래! 봄철에는 꽃구경도 하면서 등산을 즐길 수 있어.
358731,과천관이라면 가까워서 좋겠다! 응! 내년 2월 13일까지 열리니까 시간 내서 꼭 가...,맞아! 그리고 이번 전시는 최욱경 작가의 작품과 문학이 연계되어 있대
428443,2020년 젊은작가상 수상 작가 장희원의 첫번째 소설집이에요. 제목은 '우리의 환대...,마음에 온기가 삭막한 이 시대에는 그런 소설이 필요할 것 같아.


In [ ]:
analyze_top_frequent_korean(diverse_df, 'res', top_n=200)

In [ ]:
quick_token_analysis(diverse_df, 'res')

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

def cluster_diverse_sampling(df, text_col='res', n_clusters=500, target_size=50000):
    """클러스터링으로 다양성 확보"""
    
    # TF-IDF
    vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
    tfidf = vectorizer.fit_transform(df[text_col].astype(str))
    
    # 클러스터링
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42)
    df['cluster'] = kmeans.fit_predict(tfidf)
    
    # 각 클러스터에서 균등하게 샘플링
    samples_per_cluster = target_size // n_clusters
    
    sampled_dfs = []
    for cluster_id in range(n_clusters):
        cluster_df = df[df['cluster'] == cluster_id]
        n = min(samples_per_cluster, len(cluster_df))
        sampled_dfs.append(cluster_df.sample(n=n))
    
    result = pd.concat(sampled_dfs).drop('cluster', axis=1)
    
    print(f"클러스터 수: {n_clusters}")
    print(f"클러스터당: {samples_per_cluster}개")
    print(f"최종: {len(result):,}개")
    
    return result.reset_index(drop=True)

# 사용
df_cluster = cluster_diverse_sampling(df, n_clusters=500, target_size=50000)

클러스터 수: 500
클러스터당: 100개
최종: 49,908개


In [49]:
analyze_top_frequent_korean(df_cluster, 'res', top_n=200)

상위 200개 빈도 토큰 분석 (공백+한국어 포함)

=== 토큰 종류 개수 ===
한글      : 148개 ( 74.0%)
영어      :   0개 (  0.0%)
숫자      :   4개 (  2.0%)
특수문자   :  47개 ( 23.5%)
기타      :   1개 (  0.5%)

=== 실제 사용 빈도 (상위 200개 토큰의 총 출현 횟수) ===
한글      :    448,553회 (70.41%)
영어      :          0회 ( 0.00%)
숫자      :     10,487회 ( 1.65%)
특수문자   :    168,304회 (26.42%)
기타      :      9,719회 ( 1.53%)

상위 200개 토큰이 전체의 64.00% 차지

상위 200개 중 한글 토큰 전체 목록 (148개)
  1. ID  12802: '이                   ' - 15,593회
  2. ID  47985: '도                   ' - 13,852회
  3. ID  17877: '을                   ' - 11,606회
  4. ID  89659: '야                   ' - 11,543회
  5. ID  52959: '아                   ' - 9,743회
  6. ID  31079: '어                   ' - 9,477회
  7. ID  35509: '␣가                  ' - 8,424회
  8. ID  19391: '에                   ' - 8,060회
  9. ID  19969: '가                   ' - 7,582회
 10. ID  33704: '은                   ' - 6,984회
 11. ID  21329: '지                   ' - 6,971회
 12. ID  28733: '␣수                  ' - 6,774회
 13

{'korean_count': 148,
 'english_count': 0,
 'korean_freq': 448553,
 'english_freq': 0,
 'korean_tokens': [(12802, '이', 15593),
  (47985, '도', 13852),
  (17877, '을', 11606),
  (89659, '야', 11543),
  (52959, '아', 9743),
  (31079, '어', 9477),
  (35509, ' 가', 8424),
  (19391, '에', 8060),
  (19969, '가', 7582),
  (33704, '은', 6984),
  (21329, '지', 6971),
  (28733, ' 수', 6774),
  (16560, '는', 6559),
  (126352, ' 거', 6472),
  (48606, '그', 6292),
  (144410, 'ㅋ', 6064),
  (60315, '나', 5846),
  (34395, '고', 5806),
  (26698, '서', 5728),
  (66845, '대', 5682),
  (17380, '로', 5382),
  (127353, ' 있어', 5373),
  (20401, '의', 5363),
  (25715, '자', 5178),
  (56475, '에서', 5043),
  (23573, '한', 4988),
  (28002, '리', 4578),
  (80573, '와', 4263),
  (33883, '해', 4156),
  (32290, '면', 4027),
  (139772, '그래', 4004),
  (18411, '를', 3947),
  (32077, '일', 3831),
  (29326, '시', 3678),
  (31328, '인', 3414),
  (24897, '스', 3359),
  (13146, '다', 3344),
  (20487, '기', 3243),
  (60716, ' 해', 3132),
  (127816, '겠다', 3110)

In [50]:
quick_token_analysis(df_cluster, 'res')

=== res 컬럼 토큰 분석 ===
총 샘플 수: 49,908
총 토큰 수: 995,346
고유 토큰 수: 3,564
다양성 비율: 0.36%

상위 10개 토큰 집중도: 13.77%
상위 100개 토큰 집중도: 48.11%

가장 많이 사용된 토큰 20개:
  '!': 21,808회 (2.19%)
  '?': 17,328회 (1.74%)
  '.': 16,421회 (1.65%)
  '이': 15,593회 (1.57%)
  '도': 13,852회 (1.39%)
  '을': 11,606회 (1.17%)
  '야': 11,543회 (1.16%)
  '아': 9,743회 (0.98%)
  ' ': 9,719회 (0.98%)
  '어': 9,477회 (0.95%)
  ',': 9,100회 (0.91%)
  ' �': 8,948회 (0.90%)
  ' 가': 8,424회 (0.85%)
  '에': 8,060회 (0.81%)
  '가': 7,582회 (0.76%)
  '은': 6,984회 (0.70%)
  '지': 6,971회 (0.70%)
  ' 수': 6,774회 (0.68%)
  '는': 6,559회 (0.66%)
  ' 거': 6,472회 (0.65%)


Counter({0: 21808,
         30: 17328,
         13: 16421,
         12802: 15593,
         47985: 13852,
         17877: 11606,
         89659: 11543,
         52959: 9743,
         220: 9719,
         31079: 9477,
         11: 9100,
         5140: 8948,
         35509: 8424,
         19391: 8060,
         19969: 7582,
         33704: 6984,
         21329: 6971,
         28733: 6774,
         16560: 6559,
         126352: 6472,
         48606: 6292,
         144410: 6064,
         60315: 5846,
         34395: 5806,
         26698: 5728,
         66845: 5682,
         10764: 5394,
         17380: 5382,
         127353: 5373,
         20401: 5363,
         120: 5341,
         25715: 5178,
         56475: 5043,
         23573: 4988,
         28002: 4578,
         230: 4372,
         254: 4293,
         80573: 4263,
         33883: 4156,
         32290: 4027,
         139772: 4004,
         18411: 3947,
         32077: 3831,
         29326: 3678,
         112: 3636,
         15: 3431,
    

In [51]:
full_breakdown_with_english(df_cluster, 'res')

=== 사용된 토큰 전체 분류 ===
총 사용 토큰: 3,564개

한글        : 2,267개 (63.61%)
영어        :   947개 (26.57%)
숫자        :    10개 ( 0.28%)
특수문자      :   318개 ( 8.92%)
공백        :     3개 ( 0.08%)
기타        :    19개 ( 0.53%)

검증: 3564 = 3564

=== 한글 토큰 샘플 20개 ===
  ID 139269: '통신'
  ID 139274: ' 앞'
  ID 139287: '널'
  ID 131097: '새'
  ID 139293: ' 다만'
  ID 131110: '회사'
  ID 131120: '총'
  ID  98358: ' 종'
  ID 131131: '티'
  ID 139325: '었던'
  ID  32831: '성'
  ID 139328: '마다'
  ID 131137: '처럼'
  ID 139327: ' 만약'
  ID 147547: 'ㄳ'
  ID 131170: ' 다시'
  ID 131196: '브'
  ID 131193: '사항'
  ID 139393: ' 수행'
  ID  73859: ' 확인'

=== 영어 토큰 샘플 20개 ===
  ID     32: 'A'
  ID  98332: ' Choir'
  ID     35: 'D'
  ID     33: 'B'
  ID     37: 'F'
  ID     38: 'G'
  ID  65576: ' ASM'
  ID     41: 'J'
  ID     42: 'K'
  ID     43: 'L'
  ID     44: 'M'
  ID     45: 'N'
  ID     46: 'O'
  ID     39: 'H'
  ID     49: 'R'
  ID     48: 'Q'
  ID     51: 'T'
  ID     53: 'V'
  ID     55: 'X'
  ID     56: 'Y'


{'korean': 2267,
 'english': 947,
 'digit': 10,
 'special': 318,
 'space': 3,
 'etc': 19}

In [52]:
df_cluster

,req,res,entropy
0,"응, 가봤어! 한복을 입고 나온 바이올렛 씨가 인상적이었어. 한복을 입고 나왔다고?...",와 짜짜 머박이네! 나도 한국에 가보고 싶다.,2.995732
1,맞아! 그리고 합천막걸리도 핫해! 막걸리도 마시면서 맛있는 음식을 먹으면 완내스야!...,한과와 떡도 먹어보고 싶다!,2.708050
2,사람들이 밀면 빛도 내면서 울리는 이끼들이 있는 거래. 와... 신기하다! 날씨에 ...,나도 가보고 싶다! 언제 가야 될까?,2.540036
3,일정도 원하는 대로 선택할 수 있어 봄 여행에 딱이네 ㅎㅎ 일본은 오사카가 인기 1위래,나도 가보고 싶다 ㅠㅠ,2.163956
4,길이 짧고 거리도 가까워서 편할 것 같아 그래서 어떤 명소가 있는데? 오백년소나무가...,우와! 그 소나무 찾아보고 싶다,2.484907
...,...,...,...
49903,그래? 언제 시작해? 5월 15일 오후 5시부터 시작이래! 공연 내용은 무슨 이야기야?,조선 후기 풍속화 '대쾌도'를 모티브로 한 연희극이래!,3.258097
49904,"응, 가봤는데 왕실의례 전시실은 첨 들어봤어 그러면 이번에 보잠발기 전시도 GBJG...",조선 왕실 의례 때 사용한 비녀 목록을 적은 기록물이래,3.028029
49905,오우예 씨몬이네! 어떤 장식이 돋보였어? 창덕궁 인정문에서 보이는 조선 후기 기와 ...,맞아! 이건 조선 전기 왕실 관련 건축물 세부모습에 대한 실질적 고증이 가능한 유일...,3.564623
49906,그리고 경품도 푸짐하대! 경품 뽑으러 가야지 연휴에 진행되는 대표적인 행사들도 알아...,용인 한국민속촌에서 사또와 조선 캐릭터 잔치도 열려!,3.178054


In [53]:
# 혹시 안 해봤다면:
def quick_response_check(df, text_col='res'):
    total = len(df)
    unique = df[text_col].nunique()
    diversity = unique / total * 100
    
    print(f"총 샘플: {total:,}개")
    print(f"고유 응답: {unique:,}개")
    print(f"다양성: {diversity:.1f}%")
    print(f"평균 반복: {total/unique:.1f}회")

quick_response_check(df_cluster)

총 샘플: 49,908개
고유 응답: 48,339개
다양성: 96.9%
평균 반복: 1.0회


In [54]:
# 어떤 영어가 남았는지
def check_remaining_english(df, text_col='res'):
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    token_counts = Counter(all_tokens)
    
    english_tokens = []
    for token_id, count in token_counts.items():
        decoded = tokenizer.decode([token_id])
        has_korean = bool(re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        
        if has_english and not has_korean:
            english_tokens.append((token_id, decoded, count))
    
    english_tokens.sort(key=lambda x: x[2], reverse=True)
    
    print(f"=== 영어 토큰 상위 30개 ===")
    for i, (tid, text, count) in enumerate(english_tokens[:30], 1):
        display = text.replace(' ', '␣')
        print(f"{i:2d}. '{display:15s}' - {count:,}회")
    
    # GBJG, K- 남았는지 확인
    gbjg_tokens = [x for x in english_tokens if 'GBJG' in x[1] or 'BJ' in x[1]]
    k_tokens = [x for x in english_tokens if x[1].startswith('K-') or x[1].startswith(' K-')]
    
    print(f"\nGBJG 관련: {len(gbjg_tokens)}개")
    print(f"K- 관련: {len(k_tokens)}개")

check_remaining_english(df_cluster)

=== 영어 토큰 상위 30개 ===
 1. 'G              ' - 146회
 2. '␣G             ' - 139회
 3. 'BJ             ' - 129회
 4. '␣K             ' - 94회
 5. 'km             ' - 61회
 6. 'Z              ' - 58회
 7. 'K              ' - 46회
 8. '␣M             ' - 45회
 9. 'm              ' - 39회
10. '␣S             ' - 36회
11. 'V              ' - 34회
12. '␣A             ' - 30회
13. '␣DM            ' - 26회
14. '␣CG            ' - 24회
15. 'D              ' - 22회
16. '␣N             ' - 22회
17. 'FT             ' - 19회
18. 'R              ' - 18회
19. 'TV             ' - 17회
20. '␣OST           ' - 16회
21. 'BS             ' - 15회
22. 'ICE            ' - 15회
23. '␣DJ            ' - 15회
24. 'NS             ' - 14회
25. '␣D             ' - 14회
26. '␣R             ' - 14회
27. 'F              ' - 13회
28. '␣LED           ' - 13회
29. 'X              ' - 13회
30. 'M              ' - 13회

GBJG 관련: 1개
K- 관련: 0개


In [55]:
# 한글 2,267개가 뭔지 확인
def analyze_korean_tokens_detail(df, text_col='res'):
    all_tokens = []
    for text in df[text_col].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(tokens)
    
    token_counts = Counter(all_tokens)
    
    korean_tokens = []
    for token_id, count in token_counts.items():
        decoded = tokenizer.decode([token_id])
        has_korean = bool(re.search(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', decoded))
        has_english = bool(re.search(r'[a-zA-Z]', decoded))
        
        if has_korean and not has_english:
            korean_tokens.append((token_id, decoded, count))
    
    korean_tokens.sort(key=lambda x: x[2], reverse=True)
    
    print(f"=== 한글 토큰 상위 50개 ===")
    for i, (tid, text, count) in enumerate(korean_tokens[:50], 1):
        display = text.replace(' ', '␣')
        print(f"{i:2d}. '{display:15s}' - {count:,}회")

analyze_korean_tokens_detail(df_cluster)

=== 한글 토큰 상위 50개 ===
 1. '이              ' - 15,593회
 2. '도              ' - 13,852회
 3. '을              ' - 11,606회
 4. '야              ' - 11,543회
 5. '아              ' - 9,743회
 6. '어              ' - 9,477회
 7. '␣가             ' - 8,424회
 8. '에              ' - 8,060회
 9. '가              ' - 7,582회
10. '은              ' - 6,984회
11. '지              ' - 6,971회
12. '␣수             ' - 6,774회
13. '는              ' - 6,559회
14. '␣거             ' - 6,472회
15. '그              ' - 6,292회
16. 'ㅋ              ' - 6,064회
17. '나              ' - 5,846회
18. '고              ' - 5,806회
19. '서              ' - 5,728회
20. '대              ' - 5,682회
21. '로              ' - 5,382회
22. '␣있어            ' - 5,373회
23. '의              ' - 5,363회
24. '자              ' - 5,178회
25. '에서             ' - 5,043회
26. '한              ' - 4,988회
27. '리              ' - 4,578회
28. '와              ' - 4,263회
29. '해              ' - 4,156회
30. '면              ' - 4,027회
31. '그래             ' - 4,004회
32. '를        

In [67]:
check_ngram_patterns(df_cluster, 'res', n=3, top_k=20)


=== 3-gram 패턴 (상위 20개) ===
총 3-gram: 895,530
고유 3-gram: 350,816
다양성: 39.17%

' 것 같아': 2,303회
'맞아!': 2,054회
' 거야?': 1,847회
' 레': 1,746회
' 꼭': 1,390회
'그러면': 1,291회
' 곳이': 1,237회
' 싶어': 1,099회
' ㅋㅋ': 1,093회
'��알': 1,012회
' 볼 수': 989회
' 가자!': 913회
' 멋진': 883회
'고 싶': 856회
' 있을까?': 847회
'보고 싶': 825회
'이야.': 806회
' 렬': 782회
'��루': 782회
' 음식': 741회


In [58]:
check_ngram_patterns(df_cluster, 'res', n=10,top_k=20)


=== 10-gram 패턴 (상위 20개) ===
총 10-gram: 550,423
고유 10-gram: 529,834
다양성: 96.26%

' 가서 즐거운 시간 보내자!': 110회
' 꼭 가서 돈쭐내야': 87회
'�� 가서 돈쭐내야지': 87회
' 같이 가서 즐거운 시간 보내자': 83회
' 멋진 경험이 될 것 같아': 77회
' 맛있는 음식도 먹고': 74회
' 특별한 경험이 될 것 같아': 66회
'! 같이 가서 즐거운 시간 보내': 63회
'�있는 음식도 먹고 �': 45회
'있는 음식도 먹고 싶': 45회
' 꼭 읽어봐야겠어': 40회
'은 레알루다가 멋진': 36회
' 레알루다가 멋진 �': 32회
'��알루다가 멋진 곳': 32회
'�진 경험이 될 것 같아!': 32회
'� 가서 돈쭐내야지!': 31회
'에서 맛있는 음식도 먹': 30회
' 꼭 보러 가야겠어!': 29회
' 음식도 먹고 싶어': 29회
'도 꼭 읽어봐야겠': 29회


In [59]:
# 더 짧은 n-gram 분석
for n in [2, 3, 5]:
    ngrams = []
    for text in df_cluster['res'].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        for i in range(len(tokens) - n + 1):
            ngrams.append(tuple(tokens[i:i+n]))
    
    ngram_counts = Counter(ngrams)
    
    print(f"\n=== {n}-gram ===")
    print(f"다양성: {len(ngram_counts)/len(ngrams)*100:.2f}%")
    print(f"\n상위 20개:")
    for pattern, count in ngram_counts.most_common(20):
        decoded = tokenizer.decode(list(pattern))
        print(f"  '{decoded}': {count:,}회")


=== 2-gram ===
다양성: 12.55%

상위 20개:
  ' 거야': 3,471회
  '야?': 3,273회
  '아!': 3,167회
  '맞아': 2,971회
  'ㅋㅋ': 2,752회
  ' 같아': 2,644회
  '이야': 2,613회
  ' �': 2,571회
  '까?': 2,544회
  ' 것 같': 2,363회
  ' 1': 2,187회
  '자!': 2,133회
  ' 싶': 2,052회
  ' 수 있어': 2,024회
  '그래!': 2,014회
  ' 곳': 1,912회
  '그럼': 1,885회
  '야.': 1,828회
  '��': 1,746회
  ' 즐': 1,628회

=== 3-gram ===
다양성: 39.17%

상위 20개:
  ' 것 같아': 2,303회
  '맞아!': 2,054회
  ' 거야?': 1,847회
  ' 레': 1,746회
  ' 꼭': 1,390회
  '그러면': 1,291회
  ' 곳이': 1,237회
  ' 싶어': 1,099회
  ' ㅋㅋ': 1,093회
  '��알': 1,012회
  ' 볼 수': 989회
  ' 가자!': 913회
  ' 멋진': 883회
  '고 싶': 856회
  ' 있을까?': 847회
  '보고 싶': 825회
  '이야.': 806회
  ' 렬': 782회
  '��루': 782회
  ' 음식': 741회

=== 5-gram ===
다양성: 74.71%

상위 20개:
  ' 레알루': 393회
  '��알루다가': 393회
  ' 렬루다가': 363회
  ' 좋을 것 같아': 362회
  ' 즐거운 시간': 359회
  ' 레게노': 342회
  ' 꼭 가봐': 337회
  '��게노다': 307회
  '�� 가봐야': 294회
  ' 맛있는 음': 283회
  '�있는 음식': 281회
  ' 흥미로': 279회
  '�거운 시간 보내': 277회
  ' 가보고 싶어': 269회
  ' 수 있을 것 같아': 245회
  ' 읽어봐야': 238회
  '

In [60]:
singleturn_df = pd.read_csv('/data/ephemeral/pro-nlp-finalproject-nlp-13/data/persona_data/AllData.csv')

In [61]:
singleturn_df

,req,res
0,너 좋아하는 차 종류 있어?,무슨 차? 자동차? 마시는 차?
1,ㅋㅋ 마시는 차 말한 거야!,"아하 나 둥글레, 옥수수, 보리차 좋아해"
2,완전 곡물류 좋아하네 ㅋㅋ,야쓰 끓이기 귀찮아서 냉침해 먹어
3,그럼 오래 걸리지 않아?,끓이는 것보다는 훨씬 오래 걸리지 ㅠ
4,근데 냉침 하는 것도 귀찮겠다 ㅜㅠ,응! 그래서 매일은 안 먹고 가끔 마셔
...,...,...
88969,아르바이트 면접 언제야?,내일 아르바이트 면접이야!
88970,무슨 아르바이트 지원했어?,나 카페 아르바이트 지원했어
88971,카페는 경력자 원하지 않아?,맞아 나 전에 카페 경력 일년 정도 있어서 괜찮을걸?
88972,아 맞네 ㅋㅋ 너 카페 일했었네,응 근데 까먹었긴 한데 일 배우면 또 다해


In [62]:
analyze_top_frequent_korean(singleturn_df,'res', top_n=200)

상위 200개 빈도 토큰 분석 (공백+한국어 포함)

=== 토큰 종류 개수 ===
한글      : 149개 ( 74.5%)
영어      :   0개 (  0.0%)
숫자      :   0개 (  0.0%)
특수문자   :  50개 ( 25.0%)
기타      :   1개 (  0.5%)

=== 실제 사용 빈도 (상위 200개 토큰의 총 출현 횟수) ===
한글      :    721,415회 (72.87%)
영어      :          0회 ( 0.00%)
숫자      :          0회 ( 0.00%)
특수문자   :    242,411회 (24.49%)
기타      :     26,208회 ( 2.65%)

상위 200개 토큰이 전체의 65.30% 차지

상위 200개 중 한글 토큰 전체 목록 (149개)
  1. ID 144410: 'ㅋ                   ' - 42,872회
  2. ID  52959: '아                   ' - 23,209회
  3. ID  12802: '이                   ' - 21,705회
  4. ID  47985: '도                   ' - 20,812회
  5. ID  31079: '어                   ' - 19,678회
  6. ID  16560: '는                   ' - 16,304회
  7. ID  60315: '나                   ' - 15,562회
  8. ID  21329: '지                   ' - 14,969회
  9. ID  19969: '가                   ' - 14,551회
 10. ID  34395: '고                   ' - 12,852회
 11. ID  89659: '야                   ' - 12,837회
 12. ID 144065: 'ㅠ                   ' - 10,

{'korean_count': 149,
 'english_count': 0,
 'korean_freq': 721415,
 'english_freq': 0,
 'korean_tokens': [(144410, 'ㅋ', 42872),
  (52959, '아', 23209),
  (12802, '이', 21705),
  (47985, '도', 20812),
  (31079, '어', 19678),
  (16560, '는', 16304),
  (60315, '나', 15562),
  (21329, '지', 14969),
  (19969, '가', 14551),
  (34395, '고', 12852),
  (89659, '야', 12837),
  (144065, 'ㅠ', 10883),
  (19391, '에', 10797),
  (48606, '그', 9971),
  (26698, '서', 9165),
  (126352, ' 거', 8706),
  (33704, '은', 8518),
  (124419, '데', 8434),
  (92192, '거', 8288),
  (16186, '하', 8062),
  (53900, ' 하', 7981),
  (17877, '을', 7314),
  (73518, ' 나', 7242),
  (28002, '리', 7094),
  (95170, ' 안', 7092),
  (13146, '다', 7041),
  (57801, '게', 6939),
  (20487, '기', 6818),
  (33883, '해', 6725),
  (83036, '니', 6712),
  (54825, ' 그', 6705),
  (32290, '면', 6452),
  (35509, ' 가', 6431),
  (88259, '구', 6069),
  (126730, '네', 6014),
  (131518, '응', 5606),
  (17380, '로', 5441),
  (60716, ' 해', 5400),
  (25715, '자', 5041),
  (144330, '

In [63]:
quick_token_analysis(singleturn_df,'res')

=== res 컬럼 토큰 분석 ===
총 샘플 수: 88,969
총 토큰 수: 1,516,124
고유 토큰 수: 3,353
다양성 비율: 0.22%

상위 10개 토큰 집중도: 15.03%
상위 100개 토큰 집중도: 50.22%

가장 많이 사용된 토큰 20개:
  'ㅋ': 42,872회 (2.83%)
  '?': 26,572회 (1.75%)
  ' ': 26,208회 (1.73%)
  '아': 23,209회 (1.53%)
  '이': 21,705회 (1.43%)
  '도': 20,812회 (1.37%)
  '어': 19,678회 (1.30%)
  '는': 16,304회 (1.08%)
  '나': 15,562회 (1.03%)
  '지': 14,969회 (0.99%)
  '가': 14,551회 (0.96%)
  ' �': 14,263회 (0.94%)
  '고': 12,852회 (0.85%)
  '야': 12,837회 (0.85%)
  '.': 11,803회 (0.78%)
  '/': 11,672회 (0.77%)
  'ㅠ': 10,883회 (0.72%)
  '에': 10,797회 (0.71%)
  ' �': 10,040회 (0.66%)
  '그': 9,971회 (0.66%)


Counter({144410: 42872,
         30: 26572,
         220: 26208,
         52959: 23209,
         12802: 21705,
         47985: 20812,
         31079: 19678,
         16560: 16304,
         60315: 15562,
         21329: 14969,
         19969: 14551,
         5140: 14263,
         34395: 12852,
         89659: 12837,
         13: 11803,
         14: 11672,
         144065: 10883,
         19391: 10797,
         10764: 10040,
         48606: 9971,
         26698: 9165,
         126352: 8706,
         1112: 8641,
         33704: 8518,
         124419: 8434,
         92192: 8288,
         16186: 8062,
         53900: 7981,
         250: 7626,
         17877: 7314,
         73518: 7242,
         28002: 7094,
         95170: 7092,
         13146: 7041,
         57801: 6939,
         230: 6827,
         20487: 6818,
         112: 6758,
         33883: 6725,
         83036: 6712,
         54825: 6705,
         120: 6659,
         32290: 6452,
         35509: 6431,
         226: 6367,
         3

In [64]:
for n in [2, 3, 5]:
    ngrams = []
    for text in singleturn_df['res'].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        for i in range(len(tokens) - n + 1):
            ngrams.append(tuple(tokens[i:i+n]))
    
    ngram_counts = Counter(ngrams)
    
    print(f"\n=== {n}-gram ===")
    print(f"다양성: {len(ngram_counts)/len(ngrams)*100:.2f}%")
    print(f"\n상위 20개:")
    for pattern, count in ngram_counts.most_common(20):
        decoded = tokenizer.decode(list(pattern))
        print(f"  '{decoded}': {count:,}회")


=== 2-gram ===
다양성: 12.38%

상위 20개:
  'ㅋㅋ': 21,282회
  ' ㅋ': 15,380회
  'ㅠㅠ': 4,717회
  ' ㅠ': 4,521회
  '맞아': 4,454회
  ' 같아': 3,794회
  ' 진짜': 3,743회
  ' 먹': 3,412회
  '야?': 3,134회
  ' �': 3,066회
  ' 하하': 2,982회
  '어?': 2,938회
  '나는': 2,549회
  ' 싶': 2,367회
  ' 것 같': 2,366회
  ' �': 2,365회
  '나도': 2,330회
  '구나': 2,285회
  ' 뭐': 2,263회
  '?/': 2,192회

=== 3-gram ===
다양성: 44.98%

상위 20개:
  ' ㅋㅋ': 15,157회
  ' ㅠㅠ': 3,561회
  'ㅋㅋ/': 2,057회
  ' 했': 2,044회
  ' 것 같아': 1,974회
  ' 엄청': 1,577회
  ' ㅜㅜ': 1,498회
  ' 봐': 1,483회
  ' 힘': 1,360회
  ' 봤': 1,343회
  ' 근데': 1,330회
  '고 싶': 1,282회
  ' 거 같아': 1,251회
  ' 편': 1,195회
  '어 ㅋ': 1,156회
  ' 괜': 1,054회
  '아 ㅋ': 1,053회
  '��들': 1,037회
  '? ㅋ': 999회
  '��찮': 950회

=== 5-gram ===
다양성: 86.81%

상위 20개:
  '맞아 ㅋㅋ': 357회
  ' 같아 ㅋㅋ': 275회
  ' 괜찮아': 250회
  'ㅋㅋ 근데': 230회
  ' 힘들어': 217회
  ' 괜찮은': 200회
  '? ㅋㅋ/': 198회
  ' 거 같아 ㅋ': 184회
  ' 했어?': 178회
  ' 봤어?': 177회
  ' 편이야': 172회
  '그렇구나.': 167회
  '도 괜찮': 163회
  '어 ㅋㅋ/': 160회
  '구나 ㅋㅋ': 148회
  ' ㅋㅋ/그': 146회
  ' 스타일': 146회


In [66]:
for n in [4, 6, 7]:
    ngrams = []
    for text in singleturn_df['res'].dropna().astype(str):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        for i in range(len(tokens) - n + 1):
            ngrams.append(tuple(tokens[i:i+n]))
    
    ngram_counts = Counter(ngrams)
    
    print(f"\n=== {n}-gram ===")
    print(f"다양성: {len(ngram_counts)/len(ngrams)*100:.2f}%")
    print(f"\n상위 20개:")
    for pattern, count in ngram_counts.most_common(20):
        decoded = tokenizer.decode(list(pattern))
        print(f"  '{decoded}': {count:,}회")


=== 4-gram ===
다양성: 71.41%

상위 20개:
  ' ㅋㅋ/': 2,036회
  '어 ㅋㅋ': 1,142회
  '아 ㅋㅋ': 1,042회
  ' 힘들': 1,037회
  '? ㅋㅋ': 987회
  ' 괜찮': 950회
  '지 ㅋㅋ': 623회
  ' 봤어': 563회
  ' 했어': 549회
  ' ㅠㅠ/': 531회
  '네 ㅋㅋ': 501회
  '그렇구나': 481회
  '응 ㅋㅋ': 420회
  '야 ㅋㅋ': 414회
  '고 싶어': 402회
  ' 코로나': 396회
  ' 빨리': 391회
  ' 갑자기': 369회
  ' 챙겨': 359회
  '지 않아?': 357회

=== 6-gram ===
다양성: 94.41%

상위 20개:
  ' 거 같아 ㅋㅋ': 184회
  ' ㅋㅋ 근데': 135회
  ' ㅋㅋ/근데': 108회
  ' 거 같아 ㅠㅠ': 95회
  '는 편이야': 76회
  ' 챙겨 먹': 74회
  ' 봐 ㅋㅋ': 69회
  ' 괜찮아?': 64회
  ' 것 같아 ㅋㅋ': 64회
  '��어 ㅋㅋ': 60회
  ' 했어 ㅋ': 59회
  ' 갑자기 왜': 55회
  '그러게 말이야': 55회
  ' 재밌게 봤': 49회
  ' 괜찮은데': 49회
  ' 키우고 싶': 48회
  ' 괜찮았어': 48회
  ' 싶어 ㅋㅋ': 47회
  ' 편이야?': 46회
  '기로 했어': 46회

=== 7-gram ===
다양성: 97.62%

상위 20개:
  ' 했어 ㅋㅋ': 59회
  ' 봤어 ㅋㅋ': 37회
  ' 엄청 힘들': 33회
  ' 갑자기 왜?': 29회
  '아 진짜? ㅋㅋ': 26회
  '고 싶어 ㅋㅋ': 25회
  '아 ㅋㅋ 근데': 24회
  ' 재밌게 봤어': 24회
  '나 봐 ㅋㅋ': 24회
  '던 거 같아 ㅋㅋ': 24회
  ' 괜찮은 거 같': 24회
  '고 싶어 ㅠㅠ': 24회
  ' 잘 챙겨 먹': 23회
  '는 거 같아 ㅋㅋ': 23회
  'ㅋㅋㅋㅋㅋㅋㅋ': 23회
  ' 했지 ㅋ